# Hemo Invasion PDE Demo

This tutorial shows how to run `HemoInvasion3D` in TumorTwin using the same pattern as `HGG_Demo` and `TNBC_Demo`.

We include a **mini 50-day run** for quick sanity checking.

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from tumortwin.models import HemoInvasion3D, extract_trajectory_component
from tumortwin.optimizers import LMoptimizer, LMoptions
from tumortwin.postprocessing import (
    compute_total_cell_count,
    plot_cellularity_map,
    plot_imaging_summary,
    plot_patient_timeline,
    plot_predicted_TCC,
)
from tumortwin.preprocessing import ADC_to_cellularity, compute_carrying_capacity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import CropSettings, CropTarget
from tumortwin.types.hgg_data import HGGPatientData
from tumortwin.utils import days_since_first

In [ ]:
# Choose device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Resolve paths robustly (works whether cwd is repo root or tutorials/)
cwd = Path.cwd()
repo_root = cwd if (cwd / "tumortwin").exists() else cwd.parent

patient_json = repo_root / "input_files" / "HGG_demo_001" / "HGG_demo_001.json"
if not patient_json.exists():
    raise FileNotFoundError(f"Could not find patient json: {patient_json}")

crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)
patient_data = HGGPatientData.from_file(patient_json, crop_settings=crop_settings)
carrying_capacity = compute_carrying_capacity(patient_data.brainmask_image)

print(f"Patient: {patient_data.patient}")
print(f"Visits: {len(patient_data.visits)}")
print(f"Grid shape: {patient_data.brainmask_image.array.shape}")
print(f"Carrying capacity: {carrying_capacity}")

In [ ]:
# Build initial fields from first visit
visit0 = patient_data.visits[0]

cellularity0 = ADC_to_cellularity(
    visit0.adc_image,
    visit0.roi_enhance_image,
    visit0.roi_nonenhance_image,
)

initial_n = torch.from_numpy(cellularity0.array).float().to(device)
initial_m = torch.zeros_like(initial_n)
initial_s = torch.ones_like(initial_n)

# We use ROI-enhancing voxels as a proxy vessel mask for the demo.
# In a production setup, replace this with a proper vessel segmentation/mask.
vessel_mask = torch.from_numpy((visit0.roi_enhance_image.array > 0).astype(np.bool_)).to(device)

print("Initial tensors:")
print("  n:", tuple(initial_n.shape), f"[{initial_n.min().item():.3f}, {initial_n.max().item():.3f}]")
print("  m:", tuple(initial_m.shape), f"[{initial_m.min().item():.3f}, {initial_m.max().item():.3f}]")
print("  s:", tuple(initial_s.shape), f"[{initial_s.min().item():.3f}, {initial_s.max().item():.3f}]")
print("  vessel voxels:", int(vessel_mask.sum().item()))

In [ ]:
# Initialize HemoInvasion3D model with extra-conservative defaults
model = HemoInvasion3D(
    B=torch.tensor(0.010, dtype=torch.float32, device=device),
    Dn=torch.tensor(0.001, dtype=torch.float32, device=device),
    Ds=torch.tensor(0.015, dtype=torch.float32, device=device),
    k_s=torch.tensor(0.040, dtype=torch.float32, device=device),
    s_star=torch.tensor(0.250, dtype=torch.float32, device=device),
    patient_data=patient_data,
    initial_n=initial_n,
    initial_m=initial_m,
    initial_s=initial_s,
    K=torch.tensor(1.0, dtype=torch.float32, device=device),
    s_crit=torch.tensor(0.35, dtype=torch.float32, device=device),
    s_smooth=torch.tensor(0.08, dtype=torch.float32, device=device),
    s_outside=0.0,
    s_vessel=1.0,
    vessel_mask=vessel_mask,
    time_scale_days=120.0,
    poisson_iterations=32,
    require_grad=False,
    device=device,
)

solver = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=0.02),
        method="rk4",
        device=device,
        use_adjoint=False,
    ),
)

u0 = model.get_initial_state()
print("u0 shape:", tuple(u0.shape))

In [ ]:
# Mini run: 50 days (daily outputs)
mini_t0 = patient_data.visits[0].time
mini_timepoints = [mini_t0 + timedelta(days=d) for d in range(0, 51)]  # 0..50 days

times_mini, traj_mini = solver.solve(timepoints=mini_timepoints, u_initial=u0)

# Unpack fields: traj shape = (T, 3, D, H, W)
n_series = extract_trajectory_component(traj_mini, 0)
m_series = extract_trajectory_component(traj_mini, 1)
s_series = extract_trajectory_component(traj_mini, 2)

# Physical projection for diagnostics/plots (state constraints)
n_series_phys = torch.clamp(n_series, 0.0, 1.0)
m_series_phys = torch.clamp(m_series, 0.0, 1.0)
s_series_phys = torch.clamp(s_series, 0.0, 1.0)

print("Mini run complete")
print("  trajectory shape:", tuple(traj_mini.shape))
print("  n finite:", bool(torch.isfinite(n_series).all()))
print("  m finite:", bool(torch.isfinite(m_series).all()))
print("  s finite:", bool(torch.isfinite(s_series).all()))

In [ ]:
# Fast sanity checks on dynamics
mass_n = n_series_phys.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mass_m = m_series_phys.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mean_s = s_series_phys.mean(dim=(1, 2, 3)).detach().cpu().numpy()
time_days = times_mini.detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(time_days, mass_n)
axes[0].set_title("Total proliferating cells (n)")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_days, mass_m)
axes[1].set_title("Total quiescent cells (m)")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_days, mean_s)
axes[2].set_title("Mean substrate (S)")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

raw_n_min, raw_n_max = n_series.min().item(), n_series.max().item()
raw_m_min, raw_m_max = m_series.min().item(), m_series.max().item()
raw_s_min, raw_s_max = s_series.min().item(), s_series.max().item()

print("Raw ranges (before projection):")
print(f"  n: [{raw_n_min:.4f}, {raw_n_max:.4f}]")
print(f"  m: [{raw_m_min:.4f}, {raw_m_max:.4f}]")
print(f"  S: [{raw_s_min:.4f}, {raw_s_max:.4f}]")
print("Projected ranges (physical):")
print(f"  n: [{n_series_phys.min().item():.4f}, {n_series_phys.max().item():.4f}]")
print(f"  m: [{m_series_phys.min().item():.4f}, {m_series_phys.max().item():.4f}]")
print(f"  S: [{s_series_phys.min().item():.4f}, {s_series_phys.max().item():.4f}]")

# Stability warning for the raw trajectory
max_abs_raw = max(abs(raw_n_min), abs(raw_n_max), abs(raw_m_min), abs(raw_m_max))
if max_abs_raw > 5.0:
    print("WARNING: raw trajectory appears unstable (|n| or |m| > 5).")
    print("Try smaller solver step_size and/or more conservative parameters.")

In [ ]:
# Visual check: center slice for n, m, S at day 0 and day 50
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _show(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_show(axes[0, 0], n_series_phys[0, z].detach().cpu().numpy(), "n (day 0)", "magma")
_show(axes[0, 1], m_series_phys[0, z].detach().cpu().numpy(), "m (day 0)", "viridis")
_show(axes[0, 2], s_series_phys[0, z].detach().cpu().numpy(), "S (day 0)", "plasma")

_show(axes[1, 0], n_series_phys[-1, z].detach().cpu().numpy(), "n (day 50)", "magma")
_show(axes[1, 1], m_series_phys[-1, z].detach().cpu().numpy(), "m (day 50)", "viridis")
_show(axes[1, 2], s_series_phys[-1, z].detach().cpu().numpy(), "S (day 50)", "plasma")

plt.tight_layout()

## Notes

- This mini run is intended for **fast model sanity checks** only.
- For calibration/forecasting, tune parameters (`B`, `Dn`, `Ds`, `k_s`, transition parameters) and use patient-specific vascular masks if available.
- You can increase horizon and reduce `step_size` after the short run looks stable.

## Full run to last visit

This section mirrors `HGG_Demo` / `TNBC_Demo`: integrate from first to last visit with denser output.  
Then we compare the first 50 days of this full run against the mini-run.

In [ ]:
# Full run: first visit -> last visit (0.5-day output)
full_t0 = patient_data.visits[0].time
full_t1 = patient_data.visits[-1].time

full_timepoints = []
cur_t = full_t0
while cur_t <= full_t1:
    full_timepoints.append(cur_t)
    cur_t += timedelta(days=0.5)
if full_timepoints[-1] != full_t1:
    full_timepoints.append(full_t1)

times_full, traj_full = solver.solve(timepoints=full_timepoints, u_initial=u0)

n_full = extract_trajectory_component(traj_full, 0)
m_full = extract_trajectory_component(traj_full, 1)
s_full = extract_trajectory_component(traj_full, 2)

print("Full run complete")
print("  visits window (days):", (full_t1 - full_t0).days)
print("  output points:", len(full_timepoints))
print("  trajectory shape:", tuple(traj_full.shape))
print("  all finite:", bool(torch.isfinite(traj_full).all()))

In [ ]:
# Compare mini-run vs first 50 days of full-run
full_days = times_full.detach().cpu().numpy()
mini_days = times_mini.detach().cpu().numpy()

mask_50 = full_days <= 50.0
n_mass_full_50 = n_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_full_50 = m_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_full_50 = s_full[mask_50].mean(dim=(1, 2, 3)).detach().cpu().numpy()

n_mass_mini = n_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_mini = m_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_mini = s_series.mean(dim=(1, 2, 3)).detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(full_days[mask_50], n_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[0].plot(mini_days, n_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[0].set_title("Total n")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(full_days[mask_50], m_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[1].plot(mini_days, m_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[1].set_title("Total m")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(full_days[mask_50], s_mean_full_50, label="full run (<=50d)", alpha=0.9)
axes[2].plot(mini_days, s_mean_mini, "--", label="mini run 50d", alpha=0.9)
axes[2].set_title("Mean S")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()

# Visual compare center slice at ~day 50
idx50_full = int(np.argmin(np.abs(full_days - 50.0)))
idx50_mini = int(np.argmin(np.abs(mini_days - 50.0)))
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _imshow(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_imshow(axes[0, 0], n_full[idx50_full, z].detach().cpu().numpy(), "n full ~day50", "magma")
_imshow(axes[0, 1], m_full[idx50_full, z].detach().cpu().numpy(), "m full ~day50", "viridis")
_imshow(axes[0, 2], s_full[idx50_full, z].detach().cpu().numpy(), "S full ~day50", "plasma")

_imshow(axes[1, 0], n_series[idx50_mini, z].detach().cpu().numpy(), "n mini day50", "magma")
_imshow(axes[1, 1], m_series[idx50_mini, z].detach().cpu().numpy(), "m mini day50", "viridis")
_imshow(axes[1, 2], s_series[idx50_mini, z].detach().cpu().numpy(), "S mini day50", "plasma")

plt.tight_layout()

## Parameter calibration (LM)

Direct Levenberg–Marquardt calibration of the 5 key PDE parameters (`B`, `Dn`, `Ds`, `k_s`, `s_crit`) against longitudinal visit data.

The model uses `time_scale_days=30`, so parameter values are in **model units** — divide by 30 to get effective per-day rates. Starting point is the biologically motivated Variant A from the parameter analysis.

In [ ]:
# Build measured target maps at visit times
measured_cellularity_maps = [
    ADC_to_cellularity(v.adc_image, v.roi_enhance_image, v.roi_nonenhance_image)
    for v in patient_data.visits
]

n_visits_calibration = min(6, len(patient_data.visits))
target_timepoints = [v.time for v in patient_data.visits[:n_visits_calibration]]

y_target = torch.stack(
    [
        torch.from_numpy(measured_cellularity_maps[i].array).float().to(device)
        for i in range(n_visits_calibration)
    ],
    dim=0,
)


def _predict(params, timepoints=target_timepoints):
    """Update model parameters and run forward solve, return total cellularity n+m."""
    B_val, Dn_val, ks_val, s_crit_val = params
    model.B.data = torch.tensor(float(B_val), dtype=torch.float32, device=device)
    model.Dn.data = torch.tensor(float(Dn_val), dtype=torch.float32, device=device)
    model.k_s.data = torch.tensor(float(ks_val), dtype=torch.float32, device=device)
    model.s_crit.data = torch.tensor(float(s_crit_val), dtype=torch.float32, device=device)
    _, traj = solver.solve(timepoints=timepoints, u_initial=model.get_initial_state())
    n = extract_trajectory_component(traj, 0)
    m = extract_trajectory_component(traj, 1)
    return torch.clamp(n + m, 0.0, 1.0)


# Baseline SSE before calibration
params_before = torch.tensor(
    [model.B.item(), model.Dn.item(), model.k_s.item(), model.s_crit.item()],
    dtype=torch.float64,
)
with torch.inference_mode():
    pred_before = _predict(params_before)
    baseline_sse = torch.sum((pred_before - y_target) ** 2).item()

print("Calibration targets prepared")
print(f"  visits used: {n_visits_calibration}")
print(f"  target shape: {tuple(y_target.shape)}")
print(f"  baseline SSE: {baseline_sse:.4e}")

In [ ]:
# LM calibration: 4 parameters [B, Dn, k_s, s_crit]
# Ds and s_star are fixed (substrate dynamics determined separately)
bounds = torch.tensor(
    [
        [0.5,  3.0],     # B:      eff 0.017–0.100 /day
        [0.05, 1.5],     # Dn:     eff 0.002–0.050 mm²/day
        [3.0,  25.0],    # k_s:    eff 0.10–0.83 /day
        [0.10, 0.85],    # s_crit: quiescence transition threshold
    ],
    dtype=torch.float64,
)

initial_guess = torch.tensor(
    [model.B.item(), model.Dn.item(), model.k_s.item(), model.s_crit.item()],
    dtype=torch.float64,
)
print("Initial parameters:", initial_guess.tolist())

lm_options = LMoptions(
    jac_delta=0.05,
    jac_update_interval=1,
    lambda_init=1.0,
    lambda_upscale_factor=3.0,
    lambda_downscale_factor=1.5,
    max_initial_delta=0.05,
)

optim = LMoptimizer(
    model=_predict,
    bounds=bounds,
    initial_guess=initial_guess,
    y_data=y_target,
    options=lm_options,
)

n_optim_steps = 20
for i in range(n_optim_steps):
    optim.step()
    p = optim.parameters[-1]
    print(
        f"step {i+1:02d}: SSE={optim.error[-1]:.4e}  "
        f"B={p[0]:.3f} Dn={p[1]:.3f} k_s={p[2]:.2f} s_crit={p[3]:.3f}"
    )

best_parameters = optim.parameters[-1]
print("\nBest parameters:", best_parameters.tolist())

In [ ]:
# Apply best parameters and compute calibrated SSE
model.B.data = torch.tensor(float(best_parameters[0]), dtype=torch.float32, device=device)
model.Dn.data = torch.tensor(float(best_parameters[1]), dtype=torch.float32, device=device)
model.k_s.data = torch.tensor(float(best_parameters[2]), dtype=torch.float32, device=device)
model.s_crit.data = torch.tensor(float(best_parameters[3]), dtype=torch.float32, device=device)

with torch.inference_mode():
    pred_cal = _predict(best_parameters)
    cal_sse = torch.sum((pred_cal - y_target) ** 2).item()

ts = model.time_scale_days
print(f"Baseline SSE: {baseline_sse:.4e}")
print(f"Calibrated SSE: {cal_sse:.4e}")
print(f"Improvement: {(1 - cal_sse / baseline_sse) * 100:.1f}%")
print(f"\nCalibrated parameters [B, Dn, k_s, s_crit]:")
print(f"  B={best_parameters[0]:.3f}  Dn={best_parameters[1]:.3f}  k_s={best_parameters[2]:.2f}  s_crit={best_parameters[3]:.3f}")
print(f"\nEffective rates (÷ time_scale_days={ts:.0f}):")
print(f"  B_eff  = {best_parameters[0]/ts:.4f} /day  (doubling ~ {0.693*ts/best_parameters[0]:.0f} days)")
print(f"  Dn_eff = {best_parameters[1]/ts:.4f} mm²/day")
print(f"  k_s_eff = {best_parameters[2]/ts:.4f} /day")

In [ ]:
# Loss trace
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].plot(optim.error, marker="o")
axes[0].set_title("LM calibration loss (SSE)")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("SSE")
axes[0].grid(True, alpha=0.3)

# Mean tumor density: predicted vs measured at calibration visits
pred_mean = pred_cal.mean(dim=(1, 2, 3)).detach().cpu().numpy()
tgt_mean = y_target.mean(dim=(1, 2, 3)).detach().cpu().numpy()
visit_days = np.array([(t - target_timepoints[0]).days for t in target_timepoints], dtype=float)

axes[1].plot(visit_days, tgt_mean, "o-", label="measured")
axes[1].plot(visit_days, pred_mean, "s--", label="predicted (calibrated)")
axes[1].set_title("Mean cellularity at visit times")
axes[1].set_xlabel("days from first visit")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

## Before vs after calibration

Compare predicted cellularity maps at visit times: initial parameters vs calibrated.

In [ ]:
# Predicted vs measured maps at calibration visit times (center slice)
z = y_target.shape[1] // 2
n_vis = n_visits_calibration

fig, axs = plt.subplots(3, n_vis, figsize=(max(6, 2.5 * n_vis), 8))
if n_vis == 1:
    axs = axs.reshape(3, 1)

for col in range(n_vis):
    vd = (target_timepoints[col] - target_timepoints[0]).days

    # Measured
    im0 = axs[0, col].imshow(y_target[col, z].detach().cpu().numpy(), cmap="magma", vmin=0, vmax=1)
    axs[0, col].set_title(f"measured d{vd}")
    axs[0, col].axis("off")

    # Predicted (calibrated)
    im1 = axs[1, col].imshow(pred_cal[col, z].detach().cpu().numpy(), cmap="magma", vmin=0, vmax=1)
    axs[1, col].set_title(f"predicted d{vd}")
    axs[1, col].axis("off")

    # Residual
    residual = (pred_cal[col, z] - y_target[col, z]).detach().cpu().numpy()
    vabs = max(abs(residual.min()), abs(residual.max()), 0.01)
    im2 = axs[2, col].imshow(residual, cmap="coolwarm", vmin=-vabs, vmax=vabs)
    axs[2, col].set_title(f"residual d{vd}")
    axs[2, col].axis("off")

axs[0, 0].set_ylabel("measured", rotation=90, fontsize=11)
axs[1, 0].set_ylabel("predicted", rotation=90, fontsize=11)
axs[2, 0].set_ylabel("residual", rotation=90, fontsize=11)

plt.tight_layout()

In [ ]:
# TCC comparison: before vs after calibration (full visit range)
with torch.inference_mode():
    pred_before_full = _predict(params_before, timepoints=[v.time for v in patient_data.visits])
    pred_after_full = _predict(best_parameters, timepoints=[v.time for v in patient_data.visits])

tcc_measured = [measured_cellularity_maps[i].array.sum() for i in range(len(patient_data.visits))]
tcc_before = [pred_before_full[i].sum().item() for i in range(len(patient_data.visits))]
tcc_after = [pred_after_full[i].sum().item() for i in range(len(patient_data.visits))]
visit_days_all = [(v.time - patient_data.visits[0].time).days for v in patient_data.visits]

plt.figure(figsize=(8, 4))
plt.plot(visit_days_all, tcc_measured, "ko-", label="measured", linewidth=2)
plt.plot(visit_days_all, tcc_before, "b^--", label="before calibration", alpha=0.7)
plt.plot(visit_days_all, tcc_after, "rs--", label="after calibration", alpha=0.9)
plt.xlabel("days from first visit")
plt.ylabel("total tumor cell count (TCC)")
plt.title("TCC: measured vs predicted")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## TumorTwin-style TCC/map check

This reproduces the same quick check pattern from `HGG_Demo` / `TNBC_Demo`:

- total tumor cell count curve,
- cellularity maps at selected days.

In [ ]:
# Equivalent of:
# fig, ax = plt.subplots(1, 1, figsize=(5, 2))
# plot_predicted_TCC(predicted_cellularity_maps, timepoints, ax=ax)
# ... and map snapshots at [0, 50, 100, 150, 200] days.

# Use full-run tumor trajectory, projected to physical range
predicted_cellularity_maps = [torch.clamp(u_t, 0.0, 1.0) for u_t in n_full]
timepoints = full_timepoints

fig, ax = plt.subplots(1, 1, figsize=(5, 2))
plot_predicted_TCC(predicted_cellularity_maps, timepoints, ax=ax)
plt.tight_layout()

requested_days = [0, 50, 100, 150, 200]
time_days = np.array([days_since_first(t, timepoints[0]) for t in timepoints])

available_days = [d for d in requested_days if d <= time_days.max()]
if len(available_days) < len(requested_days):
    print(
        f"Requested days {requested_days}, but max simulated day is {time_days.max():.1f}. "
        f"Using {available_days} instead."
    )

fig, axes = plt.subplots(1, len(available_days), figsize=(max(5, len(available_days) * 2.2), 2))
if len(available_days) == 1:
    axes = [axes]

for i, t in enumerate(available_days):
    # robust index lookup (nearest simulated output)
    t_idx = int(np.argmin(np.abs(time_days - t)))
    plot_cellularity_map(
        predicted_cellularity_maps[t_idx], patient_data, time=int(t), ax=axes[i]
    )

plt.tight_layout()

## Short full-run (HGG-style plots)

This section runs the full framework pipeline on a **short horizon** and reproduces the core HGG-style visual checks:

- patient timeline + imaging summary,
- predicted TCC (with measured visit TCC overlay),
- predicted cellularity snapshots,
- predicted vs measured maps at visit times within the short horizon.

In [ ]:
# 1) Framework context plots (same style as HGG demo)
plot_patient_timeline(patient_data)
plot_imaging_summary(patient_data)

In [ ]:
# 2) Short full-run (small horizon, but full pipeline)
SHORT_HORIZON_DAYS = 60
SHORT_OUTPUT_STEP_DAYS = 0.5

t0 = patient_data.visits[0].time
t_end_short = t0 + timedelta(days=SHORT_HORIZON_DAYS)

timepoints_short = []
cur_t = t0
while cur_t <= t_end_short:
    timepoints_short.append(cur_t)
    cur_t += timedelta(days=SHORT_OUTPUT_STEP_DAYS)
if timepoints_short[-1] != t_end_short:
    timepoints_short.append(t_end_short)

_, traj_short = solver.solve(timepoints=timepoints_short, u_initial=model.get_initial_state())
pred_short_n = torch.clamp(extract_trajectory_component(traj_short, 0), 0.0, 1.0)  # tumor channel only

print("Short full-run complete")
print("  horizon (days):", SHORT_HORIZON_DAYS)
print("  outputs:", len(timepoints_short))
print("  pred shape:", tuple(pred_short_n.shape))
print("  finite:", bool(torch.isfinite(pred_short_n).all()))

In [ ]:
# 3) TCC plot (predicted) + measured visit TCC overlay
predicted_cellularity_maps_short = [u_t for u_t in pred_short_n]

fig, ax = plt.subplots(1, 1, figsize=(5, 2))
plot_predicted_TCC(predicted_cellularity_maps_short, timepoints_short, ax=ax)

# measured TCC at visits inside the short horizon
measured_maps_short = []
measured_days_short = []
for visit in patient_data.visits:
    d = days_since_first(visit.time, t0)
    if d <= SHORT_HORIZON_DAYS:
        cell_map = ADC_to_cellularity(
            visit.adc_image,
            visit.roi_enhance_image,
            visit.roi_nonenhance_image,
        )
        measured_maps_short.append(torch.from_numpy(cell_map.array).float())
        measured_days_short.append(d)

if measured_maps_short:
    measured_tcc = [
        compute_total_cell_count(m, carrying_capacity=carrying_capacity).item()
        for m in measured_maps_short
    ]
    ax.scatter(measured_days_short, measured_tcc, c="tab:red", s=20, label="measured visits")
    ax.legend(loc="best")

plt.tight_layout()

In [ ]:
# 4) Predicted maps at selected times (HGG-like quick panel)
requested_days = [0, 15, 30, 45, 60]
time_days_short = np.array([days_since_first(t, timepoints_short[0]) for t in timepoints_short])

fig, axes = plt.subplots(1, len(requested_days), figsize=(5, 2))
for i, d in enumerate(requested_days):
    t_idx = int(np.argmin(np.abs(time_days_short - d)))
    plot_cellularity_map(predicted_cellularity_maps_short[t_idx], patient_data, time=d, ax=axes[i])

plt.tight_layout()

In [ ]:
# 5) Predicted vs measured maps at visit times within short horizon
visit_idxs_short = [
    i for i, v in enumerate(patient_data.visits)
    if days_since_first(v.time, t0) <= SHORT_HORIZON_DAYS
]

if len(visit_idxs_short) == 0:
    print("No visit falls inside short horizon.")
else:
    fig, axs = plt.subplots(2, len(visit_idxs_short), figsize=(max(5, 2.2 * len(visit_idxs_short)), 4))
    if len(visit_idxs_short) == 1:
        axs = np.array(axs).reshape(2, 1)

    for col, vi in enumerate(visit_idxs_short):
        vd = days_since_first(patient_data.visits[vi].time, t0)
        pred_idx = int(np.argmin(np.abs(time_days_short - vd)))

        measured_map = ADC_to_cellularity(
            patient_data.visits[vi].adc_image,
            patient_data.visits[vi].roi_enhance_image,
            patient_data.visits[vi].roi_nonenhance_image,
        )

        plot_cellularity_map(predicted_cellularity_maps_short[pred_idx], patient_data, time=int(vd), ax=axs[0, col])
        plot_cellularity_map(torch.from_numpy(measured_map.array).float(), patient_data, time=int(vd), ax=axs[1, col])

    axs[0, 0].set_ylabel("pred", rotation=90)
    axs[1, 0].set_ylabel("meas", rotation=90)
    plt.tight_layout()